```
<07_wrong_images.ipynb>

제미나이 의존도: 40-50%

모델이 어떤 이미지들을 틀리는지 궁금해서 새로운 폴더 두 개를 만들었다.
rotten_but_predicted_fresh(상한 사과인데 신선한 사과라고 추측한 경우)
fresh_but_predicted_rotten(신선한 사과인데 상한 사과라고 추측한 경우)

모델의 가중치(simple_cnn_apple.pth)와 transform은 그대로 사용하였다.

반복문을 돌리면서 예측(pred)과 정답(label)이 일치하지 않는 경우
shutil() 함수를 통해 이미지를 폴더에 저장했다.
```

In [3]:
# 모델 선언 및 모델 불러오기
import torch
import torch.nn as nn
import torch.nn.functional as F
import albumentations as A
from  albumentations.pytorch import ToTensorV2

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__() # (32, 3, 128, 128) -> (32, 16, 64, 64) -> (32, 32, 32, 32) -> (32, 64, 16, 16)
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.fc = nn.Linear(64 * 16 * 16, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

print('완료1')
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
model = SimpleCNN().to(device)
model.load_state_dict(torch.load('simple_cnn_apple.pth', map_location=device))
model.eval()

transform = A.Compose([
    A.Resize(128, 128), 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])
print('완료')

완료1
완료


In [4]:
# 폴더 생성
import os
os.makedirs("images/wrong_images/fresh_but_predicted_rotten", exist_ok=True)
os.makedirs("images/wrong_images/rotten_but_predicted_fresh", exist_ok=True)

In [5]:
# 틀린 이미지 걸러내기
from PIL import Image
import shutil
import numpy as np
import cv2

categories = ['fresh_apple', 'rotten_apple']

path = "images/more_data"

for category in categories:
    dest_path = ""
    if category == "fresh_apple":
        dest_path = "images/wrong_images/fresh_but_predicted_rotten"
    elif category == "rotten_apple":
        dest_path = "images/wrong_images/rotten_but_predicted_fresh"
    folder_path = os.path.join(path, category)
    for image in os.listdir(folder_path):
        if image.startswith('.'):
            print(os.path.join(folder_path, image))
            continue
        full_path = os.path.join(folder_path, image)
        pil_image = Image.open(full_path).convert('RGB')

        image_np = np.array(pil_image)
        image_cv = cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB)
        transformed = transform(image=image_cv)
        input_tensor = transformed['image'].unsqueeze(0)

        with torch.no_grad():
            output = model(input_tensor.to(device))
            pred = output.argmax(1)
            
        pred_text = categories[pred]

        # 정답일 때
        if category == pred_text:
            continue
        elif category != pred_text:
            shutil.copy(os.path.join(folder_path, image), os.path.join(dest_path, image))

images/more_data/fresh_apple/.ipynb_checkpoints
images/more_data/rotten_apple/.ipynb_checkpoints
